# 🖼️ 04-LLM · 25 VLM 实战与多模态对齐  > 关键词：ViT、CLIP、投影层、图文拼接、对比学习  > **纯代码实战**：不加载真实模型、不调 API，用 numpy 从零实现 VLM 的核心组件——patch 嵌入、图像文本对比对齐、可视化-语言投影与拼接生成。所有数字可复核、可手算。

## 本节要解决的问题  | # | 问题 | 对应内容 | |---|---|---| | 1 | 一张图怎么变成 LLM 认识的 token？ | §1 ViT patch 嵌入 | | 2 | 图像文本怎么"对齐"到同一空间？ | §2 CLIP 对比学习 ✍️ | | 3 | LLaVA 怎么把视觉特征喂进 LLM？ | §3 投影层 + 手算 ✍️ | | 4 | 图文拼接后怎么自回归生成？ | §4 端到端小演示 |

## 故事引入  GPT 只看得懂字，看不懂图。VLM（视觉语言模型）在 LLM 外面加了两样东西：**眼睛**（视觉编码器）和**翻译器**（投影层）。眼睛把图片切成 patch 变成向量，翻译器把视觉向量"翻译"成 LLM 的词向量空间——翻译得对不对，靠 **CLIP 对比学习**训练。本章用几十行 numpy 把这三件套全部跑通。

## 核心概念：VLM 三大件  | 组件 | 作用 | 代表 | |---|---|---| | 视觉编码器 | 图像 → patch 向量序列 | ViT（196+1 tokens） | | 投影层 | 视觉特征 → LLM 词向量空间 | LLaVA MLP（2 层） | | 对齐训练 | 图文向量拉近 | CLIP 对比损失 |   **训练范式**：① CLIP/ALIGN 做**对比预训练**（图文配对，先对齐两个模态）；② LLaVA 冻结视觉塔，训练投影层接 SFT 指令数据；③ 全量微调解锁 LLM 骨干。  **token 规模速记**：ViT-B/16 处理 224×224 → 16×16 patch → 196 patch + 1 CLS = **197 tokens**。

## 代码实战 1：ViT Patch Embedding（从零）  把 6×6 像素的"图像"切成 2×2 patch，每个 patch 展平 + 线性投影成 4 维 token，再拼上 CLS 前缀，得到模型实际的输入序列。

In [ ]:
# ---- ViT Patch Embedding：6x6 图 -> 9 patch tokens ----
import numpy as np

img = np.random.default_rng(0).integers(0, 10, (6, 6)).astype(float)  # 6x6 灰度图
P = 2                                     # patch 大小
n_patch = (6 // P) ** 2                   # 3x3 = 9 patches
patches = img.reshape(6 // P, P, 6 // P, P).transpose(0, 2, 1, 3).reshape(n_patch, P * P)
E = np.random.default_rng(1).standard_normal((P * P, 4))   # 线性投影 4->4 维
tokens = patches @ E                      # (9, 4)
cls = np.zeros((1, 4))                    # CLS token
seq = np.concatenate([cls, tokens], axis=0)
print('patches 数 =', n_patch, '| 每 patch 维度 =', P*P)
print('输入序列形状 =', seq.shape, '（1 CLS + 9 patch tokens）')
assert n_patch == 9 and seq.shape == (10, 4)
print('✓ 图像已变成 LLM 的输入 token 序列')


## 深入原理：CLIP 对比对齐  CLIP 把图像编码器和文本编码器输出投影到**同一空间的归一化向量**，用 InfoNCE 式对比损失训练：  - 正样本：配对的 (图像, 文本)，相似度 $s_{ii}$ 应大  - 负样本：batch 内其他 N−1 个文本，$s_{ij}\,(i\ne j)$ 应小  - 损失（每张图像一行 softmax 交叉熵）  $$\mathcal{L}=-\frac{1}{N}\sum_{i=1}^{N}\log\frac{e^{s_{ii}}}{\sum_{j=1}^{N}e^{s_{ij}}}$$  温度系数 $\tau$ 缩放相似度：$s_{ij}=\cos(z_i^I,z_j^T)/\tau$（§20-多模态 mm00 有 InfoNCE 手算）。

## ✍️ 手算 CLIP 对比损失：N=2 完美对齐 vs 完全反配  **设定**：$N=2$，相似度矩阵 $S=\begin{bmatrix}s_{11}&s_{12}\\s_{21}&s_{22}\end{bmatrix}$，对角为目标对。  **情形 A：完美对齐** $S=\begin{bmatrix}1&0\\0&1\end{bmatrix}$  $$\text{loss}=-\frac12\left[\log\frac{e^1}{e^1+e^0}+\log\frac{e^0}{e^0+e^1}\right]=-\log\frac{e}{e+1}=\mathbf{0.3133}$$  （两行对称，各贡献 $\log(1+e^{-1})=0.3133$）  **情形 B：完全反配** $S=\begin{bmatrix}0&1\\1&0\end{bmatrix}$  $$\text{loss}=-\log\frac{1}{1+e}=\mathbf{1.3133}\ (=\log(1+e))$$  **手算收获**：  - 对比损失 = "让对角线概率大"：完美对齐时每个正对站 ~73% 概率（$e/(e+1)$），loss≈0.31；反配时只有 ~27%（$1/(1+e)$），loss≈1.31  - 数值跨度：从 0.31 到 1.31——**同一 batch 内负样本越多，任务越难，loss 越高**（N 大时分布更平坦）  - 温度 $\tau$ 的作用：$s/\tau$——τ 小则分布尖锐（更"自信"），τ 大则平坦；§20-多模态 mm00 手算过 τ=0.5 vs 1.0 的差异  - 面试速答：CLIP 用对比学习而非生成式，因为**配对监督比像素重建便宜**——互联网上图文对海量

In [ ]:
# ---- §2.1 验证：numpy 复算 CLIP 对比损失 ----
import numpy as np

def clip_loss(S):
    e = np.exp(S - S.max(axis=1, keepdims=True))
    p = e / e.sum(axis=1, keepdims=True)
    return -np.log(np.diag(p)).mean()

S_perfect = np.array([[1.0, 0.0], [0.0, 1.0]])
S_bad = np.array([[0.0, 1.0], [1.0, 0.0]])
l_ok, l_bad = clip_loss(S_perfect), clip_loss(S_bad)
print(f'完美对齐 loss = {l_ok:.4f}，完全反配 loss = {l_bad:.4f}')
assert abs(l_ok - np.log(1 + np.exp(-1))) < 1e-9
assert abs(l_bad - np.log(1 + np.exp(1))) < 1e-9
print('\n✓ 与手算一致')


## 代码实战 2：CLIP 对比学习端到端（小规模）  真正跑一次对齐：batch 里 4 个 (图像, 文本) 对，编码器就是两个随机矩阵，训练 200 步后看相似度矩阵对角是否"发亮"。

In [ ]:
# ---- CLIP 风格对比训练：4 对图文 ----
import numpy as np

rng = np.random.default_rng(7)
N, d = 4, 8
imgs = rng.standard_normal((N, d))
txts = rng.standard_normal((N, d)) * 0.3 + imgs * 0.7   # 文本与配对图像相关
Wi = rng.standard_normal((d, d)) / np.sqrt(d)           # 图像编码器(线性)
Wt = rng.standard_normal((d, d)) / np.sqrt(d)           # 文本编码器(线性)
def normalize(z): return z / np.linalg.norm(z, axis=1, keepdims=True)
def clip_loss(S, tau=0.07):
    e = np.exp(S / tau)
    p = e / e.sum(axis=1, keepdims=True)
    return -np.log(np.diag(p)).mean()

# 朴素梯度下降训练
Wi = rng.standard_normal((d, d)) / np.sqrt(d)
Wt = rng.standard_normal((d, d)) / np.sqrt(d)
for step in range(200):
    zi = normalize(imgs @ Wi); zt = normalize(txts @ Wt)
    S = zi @ zt.T
    p = np.exp(S) / np.exp(S).sum(axis=1, keepdims=True)
    p[np.arange(N), np.arange(N)] -= 1        # dL/dS = (p - onehot)/tau
    dS = p / 0.07
    dzi = dS @ zt / np.sqrt(d); dzt = dS.T @ zi / np.sqrt(d)
    Wi -= 0.1 * imgs.T @ (dzi - zi * (zi * dzi).sum(axis=1, keepdims=True))
    Wt -= 0.1 * txts.T @ (dzt - zt * (zt * dzt).sum(axis=1, keepdims=True))
S_final = zi @ zt.T
print('训练后相似度矩阵（应近似单位阵 I）：')
print(np.round(S_final, 3))
diag_mean = np.diag(S_final).mean()
off_mean = (S_final - np.diag(np.diag(S_final))).mean()
print(f'对角均值 {diag_mean:.3f} vs 非对角均值 {off_mean:.3f}')
assert diag_mean > off_mean + 0.5
print('✓ 对比训练把图文对在共享空间中对齐')


## ✍️ 手算投影层：LLaVA 的"翻译器"  LLaVA 用一个 2 层 MLP 把视觉特征投影到 LLM 词向量空间：$z=W_2\,\text{GELU}(W_1 x+b_1)+b_2$。这里手算**单层线性投影** $y=Wx+b$。  **设定**：$x=[1.0,\ -0.5,\ 0.2]$（3 维 patch 特征），$W=\begin{bmatrix}1&0&-1\\0&1&1\end{bmatrix}$，$b=[0.1,\ -0.2]$。  $$y_0=1\cdot1+0\cdot(-0.5)+(-1)\cdot0.2+0.1=1-0.2+0.1=\mathbf{0.9}$$  $$y_1=0\cdot1+1\cdot(-0.5)+1\cdot0.2-0.2=-0.5+0.2-0.2=\mathbf{-0.5}$$  **结果**：$y=[0.9,\ -0.5]$——3 维视觉特征被"翻译"成 2 维 LLM 空间向量。  **手算收获**：  - 投影层本质 = **坐标系变换**：视觉塔输出在"图像特征空间"，LLM 词向量在"文本语义空间"，线性映射就是两个空间的翻译器  - LLaVA 真实参数：$1024\to4096\to4096$，参数量 $=1024\times4096+4096+4096\times4096+4096=\mathbf{20{,}979{,}712}\approx21$M——**一个 21M 的小模块让 LLM "看得见图"**  - 训练策略（LLaVA-1.5）：冻结 ViT 与 LLM，只训投影层（第 1 阶段）+ 全量 SFT（第 2 阶段）——**投影层学到"把图像说成 LLM 听得懂的话"**  - 面试速答：为什么需要投影层而不是直接喂？因为视觉特征与词向量空间不对齐，线性/MLP 投影成本最低且已验证有效

In [ ]:
# ---- §3.1 验证：numpy 复算投影层 ----
import numpy as np
W = np.array([[1.0, 0.0, -1.0], [0.0, 1.0, 1.0]])
b = np.array([0.1, -0.2])
x = np.array([1.0, -0.5, 0.2])
y = W @ x + b
print('y = Wx+b =', np.round(y, 4).tolist())
assert np.allclose(y, [0.9, -0.5])
p1 = 1024*4096 + 4096
p2 = 4096*4096 + 4096
print(f'LLaVA projector 参数量 = {p1} + {p2} = {p1+p2} (≈21M)')
assert p1 + p2 == 20_979_712
print('\n✓ 与手算一致')


## 代码实战 3：LLaVA 式端到端小演示  完整走一遍：4 个图像 patch → 投影 → 与 2 个文本 token 拼接 → 单头自注意力 → 输出下一步 token 的分布。这就是 VLM 推理的最简骨架。

In [ ]:
# ---- LLaVA 式图文拼接 + 自注意力（numpy 小演示） ----
import numpy as np

rng = np.random.default_rng(42)
patch_feat = rng.standard_normal((4, 6))     # 视觉塔输出: 4 patch x 6 维
Wp = rng.standard_normal((6, 8))             # 投影层: 6 -> 8 (LLM 空间)
img_tok = patch_feat @ Wp
txt_tok = rng.standard_normal((2, 8))        # 文本 token(2 x 8)
seq = np.concatenate([img_tok, txt_tok], axis=0)   # 6 tokens: 4 图 + 2 文

d = seq.shape[1]
Wq = rng.standard_normal((8, 8)) / np.sqrt(8)
Wk = rng.standard_normal((8, 8)) / np.sqrt(8)
Wv = rng.standard_normal((8, 8)) / np.sqrt(8)
Q, K, V = seq @ Wq, seq @ Wk, seq @ Wv
A = np.exp(Q @ K.T / np.sqrt(d))
A = A / A.sum(axis=1, keepdims=True)
out = A @ V

logits = out[-1] @ rng.standard_normal((8, 4))   # 输出头: 4 词表
probs = np.exp(logits - logits.max())
probs = probs / probs.sum()
print(f'输入序列: {seq.shape[0]} tokens（4 图 + 2 文），注意力权重 A 形状 {A.shape}')
print(f'最后位置输出 token 概率分布: {np.round(probs, 3).tolist()}')
print(f'采样结果: 词表第 {np.argmax(probs)} 号（argmax 解码）')
assert A.shape == (6, 6) and probs.shape == (4,)
assert abs(probs.sum() - 1) < 1e-9
print('✓ VLM 推理骨架跑通：图 patch -> 投影 -> 拼接 -> 注意力 -> 采样 token')


## 面试考点  **Q1：VLM 怎么把图像变成 LLM 输入？**  ① ViT 分 patch 成序列；② 投影层映射到词向量维；③ 与文本 token 拼接进同一 transformer。图像 token 可前置或交错（Llama 3.2 支持任意位置）。  **Q2：CLIP 和 LLaVA 什么关系？**  CLIP 是对比预训练（产出对齐的视觉塔与文本塔）；LLaVA 复用它已经对齐的视觉表示 + 投影层接 LLM——CLIP 是"地基"，LLaVA 是"接上 LLM"的用法。  **Q3：为什么投影层只有 21M 却能工作？**  视觉塔输出已被 CLIP 对齐到文本语义空间，投影只需要很小的线性校正——冻结训练也够（LLaVA-1.5 的结论）。  **Q4：图像 token 数多会不会撑爆上下文？**  会：336×336 → 576 tokens。方案：perceiver 压缩（Flamingo/Q-Former 到 32 tokens）、AnyRes 动态分块、vision token 降采样。  **Q5：对比损失为什么要温度？**  τ→0 分布尖锐（惩罚负样本更狠，要求高置信对齐），τ 大则宽容；τ=0.07 是 CLIP 调出来的经验值。

## 小结与下一章预告  - VLM = 眼睛（ViT）+ 翻译器（投影层）+ 大脑（LLM），三个组件本章全部 numpy 跑通  - 手算成对验证：CLIP 对比损失 0.3133/1.3133、投影层 $[0.9,-0.5]$、projector 参数量 20.98M  - 生成骨架：patch → 投影 → 拼接 → 注意力 → 采样首 token，一图流的完整骨架  - 更深衔接：§20-多模态（CLIP/ViT/BLIP 手算）、§15-RAG 的图文检索、MiniMind（§09）的架构——VLM 就是"LLM 架构 + 视觉输入适配

## ✅ 自测清单  - [ ] 能说出 VLM 三大件与训练两阶段  - [ ] 能手算 CLIP 对比损失（完美对齐/反配）  - [ ] 能手算投影层 Wx+b 与 LLaVA projector 参数量  - [ ] 能画出 图 patch → 投影 → 拼接 → 自注意力的推理链  - [ ] 知道 CLIP 与 LLaVA 的关系及 Q-Former 压缩动因